# Step 2 – Heating network expansion (DHNx)
**MJ2505 Project 2 – District heating network optimisation**

| | |
|---|---|
| **Goal** | Expand Step 1 to a larger neighbourhood: **10 consumers** (keep the 5 from Step 1 + 5 new), **5 producers** (keep 2 + add 3, one must be a **heat pump**), **5 pipe types** DN-25/30/40/50/60. |
| **Reads** | `data/step1/` (Step 1 network) · `results/step1/summary.csv` · demand profiles |
| **Writes** | `data/step2/` (DHNx input) · `results/step2/` – incl. **`network_losses.csv`, used by Step 4** |
| **Grading** | D – verify feasibility; *quantify* how topology, pipe selection, investment/operating cost and losses change vs. Step 1; identify underused producers / binding pipe capacities |
| **Tutorial reference** | `DHNx Tutorial/DHNx.ipynb` → *discrete_DN_numbers* |

Run `step1_base_network.ipynb` first. Legend: 🔧 ASSUMPTION · ✏️ TODO.

## 0. Imports
(The environment is created by the setup cell at the top of Step 1.)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import dhnx
import pandas as pd
import numpy as np
import time

from helpers import (DATA, RESULTS, SOLVER, SOLVER_OPTIONS, DELTA_T, INTEREST_RATE, PIPE_LIFETIME, epc,
                     load_demand_profiles, pipe_capacity_kW, write_dhnx_input, plot_network)

STEP1 = DATA / "step1" / "twn_data"
OUT = RESULTS / "step2"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Diagnostic (from the course "Prerequisites" document): which Python / packages / solver are used?
import sys, shutil
import pandas as pd
import oemof.solph as solph
import dhnx
from helpers import SOLVER

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("DHNx version:     ", dhnx.__version__)
print("oemof.solph:      ", solph.__version__)
print("pandas:           ", pd.__version__)
print("Solver used:      ", SOLVER, "->", shutil.which("cbc") or "HiGHS via the 'highspy' pip package")

## 1. Consumers: keep Step 1's five, add five new profiles
The Step 1 consumers are read from `data/step1/twn_data/consumers.csv` (so they are guaranteed identical) and five new ones are appended.

✏️ **TODO – choose and justify the five additional profiles** (larger neighbourhood → e.g. bigger buildings / different load shapes). Check the peak against the largest pipe (DN-60) capacity in section 2.

In [ ]:
demand = load_demand_profiles()                        # kW

consumers_step1 = pd.read_csv(STEP1 / "consumers.csv")

# 🔧 ASSUMPTION - five additional profiles and their positions (placeholder)
STEP2_NEW_LOADS = ["Load 3", "Load 11", "Load 17", "Load 23", "Load 29"]    # consumer 5..9
consumers_new = pd.DataFrame({
    "id":      [5,   6,   7,   8,   9],
    "lon":     [500, 560, 100, 200, 300],
    "lat":     [-60, 100, 310, 310, 310],
    "profile": STEP2_NEW_LOADS,
})
consumers_new["P_heat_max"] = demand[STEP2_NEW_LOADS].max().round(1).values

consumers = pd.concat([consumers_step1, consumers_new], ignore_index=True)
STEP2_LOADS = list(consumers["profile"])

print(consumers)
print(f"\nSum of peaks: {consumers['P_heat_max'].sum():.0f} kW | peak of summed load: "
      f"{demand[STEP2_LOADS].sum(axis=1).max():.0f} kW | annual demand: {demand[STEP2_LOADS].sum().sum()/1000:.0f} MWh")

## 2. Five pipe types
Same method and columns as Step 1 (see Step 1, section 3). The DN-30 and DN-50 rows **must keep the Step 1 values** – the cell checks this automatically.

🔧 **ASSUMPTIONS:** `v_max`, €/m and W/m for DN-25, DN-40, DN-60 (cite).

In [ ]:
# 🔧 ASSUMPTIONS - pipe catalogue (placeholders, replace with cited values)
pipe_catalog = pd.DataFrame({
    "label_3":          ["DN-25", "DN-30", "DN-40", "DN-50", "DN-60"],
    "dn_mm":            [25,      30,      40,      50,      60],
    "v_max":            [1.0,     1.1,     1.3,     1.5,     1.7],      # m/s
    "invest_eur_per_m": [450,     470,     520,     560,     600],      # €/m
    "loss_W_per_m":     [7.7,     8.3,     9.0,     9.8,     10.6],     # W/m
})
pipe_catalog["cap_max_kW"] = pipe_capacity_kW(pipe_catalog["dn_mm"], pipe_catalog["v_max"], DELTA_T).round(1)
pipe_catalog["annualised_eur_per_m_a"] = epc(pipe_catalog["invest_eur_per_m"], i=INTEREST_RATE, n=PIPE_LIFETIME).round(2)

pipe_types = pd.DataFrame({
    "label_3":      pipe_catalog["label_3"],
    "active":       1,
    "nonconvex":    1,
    "l_factor":     0,
    "l_factor_fix": pipe_catalog["loss_W_per_m"] / 1000,
    "cap_max":      pipe_catalog["cap_max_kW"],
    "cap_min":      0,
    "capex_pipes":  1e-4,
    "fix_costs":    pipe_catalog["annualised_eur_per_m_a"],
})

# consistency check with Step 1
step1_types = pd.read_csv(DATA / "step1" / "invest_data" / "network" / "pipes.csv").set_index("label_3")
pd.testing.assert_frame_equal(pipe_types.set_index("label_3").loc[step1_types.index], step1_types, check_dtype=False)
print("✅ DN-30 / DN-50 identical to Step 1")
pipe_catalog

## 3. Producers: keep P0, P1 and add three (one heat pump)
P0 and P1 are read from Step 1. 🔧 For the new producers set technology, location, design-hour capacity and marginal heat cost.

Placeholder technologies (they are the same five units modelled in oemof in Step 3):

| id | technology | note |
|---|---|---|
| P0 | biomass boiler | from Step 1 |
| P1 | gas boiler | from Step 1 |
| P2 | **heat pump** (required) | 🔧 heat source? (sewage / ambient / industrial waste heat) |
| P3 | electric boiler | 🔧 |
| P4 | solar thermal | 🔧 ✏️ discuss: at the winter design hour solar output is ~0 – you may set its capacity to 0 here |

The operating cost is converted to €/(kW·a) with the full-load hours of the **new** 10-consumer demand (same method as Step 1).

In [ ]:
FLH = demand[STEP2_LOADS].sum().sum() / demand[STEP2_LOADS].sum(axis=1).max()   # h/a

producers_step1 = pd.read_csv(STEP1 / "producers.csv")
producers_new = pd.DataFrame({
    "id":         [2,           3,                 4],
    "lon":        [600,         400,               0],
    "lat":        [0,           250,               250],
    "active":     1,
    "technology": ["heat pump", "electric boiler", "solar thermal"],
    "op_cost_eur_per_MWh":          [25,  70,  5],     # 🔧 €/MWh heat
    "heat.source.nominal_capacity": [600, 400, 150],   # 🔧 kW at the design hour
})
producers = pd.concat([producers_step1, producers_new], ignore_index=True)
producers["heat.source.variable_costs"] = producers["op_cost_eur_per_MWh"] / 1000 * FLH   # €/(kW·a)

print(f"Full-load hours of the Step 2 demand: {FLH:.0f} h/a")
print(f"Total producer capacity {producers['heat.source.nominal_capacity'].sum()} kW vs. design demand "
      f"{consumers['P_heat_max'].sum():.0f} kW (+ losses)")
producers

## 4. Forks and candidate pipes: Step 1 network + extension
Forks and pipes of Step 1 are kept, new streets and connections are appended (ids continue).
✏️ **TODO:** design your extension; include alternative routes so the optimizer has real choices.

In [ ]:
forks_step1 = pd.read_csv(STEP1 / "forks.csv")
forks_new = pd.DataFrame({
    "id":  [5,   6,   7,   8,   9],
    "lon": [500, 500, 300, 100, 200],
    "lat": [0,   100, 250, 250, 250],
})
forks = pd.concat([forks_step1, forks_new], ignore_index=True)

pipes_step1 = pd.read_csv(STEP1 / "pipes.csv")[["id", "from_node", "to_node"]]
new_edges = [
    ("producers-1", "forks-5"), ("forks-5", "forks-6"), ("forks-4", "forks-6"),             # east extension
    ("forks-4", "forks-7"), ("forks-3", "forks-8"), ("forks-8", "forks-9"), ("forks-9", "forks-7"),  # north extension
    ("producers-2", "forks-5"), ("producers-3", "forks-7"), ("producers-4", "forks-8"),      # new plants
    ("forks-5", "consumers-5"), ("forks-6", "consumers-6"), ("forks-8", "consumers-7"),      # new houses
    ("forks-9", "consumers-8"), ("forks-7", "consumers-9"),
]
pipes_new = pd.DataFrame({"id": range(len(pipes_step1), len(pipes_step1) + len(new_edges)),
                          "from_node": [a for a, _ in new_edges],
                          "to_node":   [b for _, b in new_edges]})
pipes = pd.concat([pipes_step1, pipes_new], ignore_index=True)

twn_data_path, invest_data_path = write_dhnx_input(DATA / "step2", consumers, producers, forks, pipes, pipe_types)
print(f"{len(pipes)} candidate pipes, written to {twn_data_path.parent}")

## 5. Load into DHNx and plot the given network

In [ ]:
# Initialize thermal network
network = dhnx.network.ThermalNetwork()
network = network.from_csv_folder(str(twn_data_path))
invest_opt = dhnx.input_output.load_invest_options(str(invest_data_path))

plot_network(network, "Given network (all candidate pipes) – Step 2")
plt.savefig(OUT / "network_given.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Optimise
Same problem formulation as Step 1 (section 6 there), now with 5 pipe types → 5 binary variables per candidate edge and direction, so the MILP is larger.
For **computational complexity** we record the number of binary decisions and the solve time and compare with Step 1.

If CBC stops with `maxEvaluations`, add `solver_cmdline_options={"maxNodes": 2000}` (see course prerequisites, item 13).

In [ ]:
t0 = time.perf_counter()
network.optimize_investment(invest_options=invest_opt, solver=SOLVER,
                             solve_kw={"tee": False}, solver_cmdline_options=SOLVER_OPTIONS)
solve_time = time.perf_counter() - t0
print(f"Solved in {solve_time:.1f} s")

## 7. Results

### 7.1 Selected pipes and feasibility check
Feasibility = the solver returned an optimal solution **and** every consumer is connected **and** no shortage was needed (the DHNx shortage source has a very high cost, so any use of it would show up as a huge objective value).

*Note:* with HiGHS, DHNx may log `Investment status of pipe id X is 1 for both directions!`. This comes from solver round-off (binary values such as 1.0 + 1e-14), not from a pipe really being built twice – the check below and the `direction` column confirm the solution.

In [ ]:
results_edges = network.results.optimization["components"]["pipes"]
print(results_edges[["from_node", "to_node", "length", "hp_type", "capacity", "direction", "costs", "losses"]])
objective = network.results.optimization["oemof_meta"]["objective"]
print("\nObjective value: ", objective)

built = results_edges[results_edges["capacity"] > 0.001].copy()
built = built.merge(pipe_catalog[["label_3", "cap_max_kW", "invest_eur_per_m"]],
                    left_on="hp_type", right_on="label_3", how="left").drop(columns="label_3")
built["utilisation [%]"] = (100 * built["capacity"] / built["cap_max_kW"]).round(1)

connected = {n for n in built["to_node"] if n.startswith("consumers")}
assert len(connected) == len(consumers), f"Not all consumers connected: {connected}"
print(f"✅ All {len(consumers)} consumers connected; pipe types used: {sorted(built['hp_type'].unique())}")

In [ ]:
twn_results = network
twn_results.components["pipes"] = results_edges[results_edges["capacity"] > 0.001]

plot_network(twn_results, "Result network – Step 2", show_dn=True)
plt.savefig(OUT / "network_result.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 Producers – are some much less used than others?
Utilisation = supplied heat at the design hour / producer capacity. A producer can be underused because it is expensive (merit order), far away (pipe cost), or because the pipes connecting it are at their limit.

In [ ]:
oemof_results = network.results.optimization["oemof"]
producer_supply = pd.Series({
    "P" + str(k[0]).split("producers-")[-1]: v["sequences"]["flow"].iloc[0]
    for k, v in oemof_results.items() if "source_producers" in str(k[0])
}, name="heat at design hour [kW]").sort_index()
producer_table = producer_supply.to_frame()
producer_table["capacity [kW]"] = producers.set_index("id")["heat.source.nominal_capacity"].values
producer_table["utilisation [%]"] = (100 * producer_table.iloc[:, 0] / producer_table["capacity [kW]"]).round(1)
producer_table["op cost [EUR/MWh]"] = producers["op_cost_eur_per_MWh"].values
producer_table["technology"] = producers["technology"].values
producer_table

### 7.3 Binding pipe capacities
Pipes running at (almost) their maximum capacity – these limit where heat can come from and can force the use of a more expensive producer or a larger DN.

In [ ]:
binding = built[built["utilisation [%]"] >= 99][["from_node", "to_node", "hp_type", "capacity", "cap_max_kW", "utilisation [%]"]]
print("Pipes at ≥ 99 % of capacity:" if len(binding) else "No pipe is at its capacity limit.")
binding

## 8. Compare with Step 1 (quantify, don't only describe)

In [ ]:
summary = pd.Series({
    "consumers": len(consumers),
    "producers": len(producers),
    "pipe types available": len(pipe_types),
    "candidate pipes": len(pipes),
    "built pipes": len(built),
    "built length [m]": built["length"].sum(),
    "pipe investment [EUR]": (built["length"] * built["invest_eur_per_m"]).sum(),
    "annual pipe cost [EUR/a]": built["costs"].sum(),
    "producer operating cost [EUR/a]": objective - built["costs"].sum(),
    "objective [EUR/a]": objective,
    "design demand [kW]": consumers["P_heat_max"].sum(),
    "heat losses [kW]": built["losses"].sum(),
    "heat losses [MWh/a]": built["losses"].sum() * 8760 / 1000,
    "annual demand [MWh/a]": demand[STEP2_LOADS].sum().sum() / 1000,
    "solve time [s]": solve_time,
})
summary["losses / demand [%]"] = 100 * summary["heat losses [MWh/a]"] / summary["annual demand [MWh/a]"]
summary["binary pipe decisions (≈ 2 directions × types × candidate pipes)"] = 2 * len(pipe_types) * len(pipes)

step1 = pd.read_csv(RESULTS / "step1" / "summary.csv", index_col=0)["Step 1"]
step1["binary pipe decisions (≈ 2 directions × types × candidate pipes)"] = 2 * step1["pipe types available"] * step1["candidate pipes"]
comparison = pd.DataFrame({"Step 1": step1, "Step 2": summary})
comparison["change [%]"] = 100 * (comparison["Step 2"] / comparison["Step 1"] - 1)
comparison["Step 2 per kW design demand"] = np.nan
for k in ["pipe investment [EUR]", "annual pipe cost [EUR/a]", "heat losses [kW]", "built length [m]"]:
    comparison.loc[k, "Step 2 per kW design demand"] = summary[k] / summary["design demand [kW]"]
    comparison.loc[k, "Step 1 per kW design demand"] = step1[k] / step1["design demand [kW]"]
comparison.round(3)

In [ ]:
# pipe-type mix: Step 1 vs Step 2 (metres built per DN)
step1_pipes = pd.read_csv(RESULTS / "step1" / "pipes_results.csv")
mix = pd.DataFrame({"Step 1 [m]": step1_pipes.groupby("hp_type")["length"].sum(),
                    "Step 2 [m]": built.groupby("hp_type")["length"].sum()}).reindex(pipe_catalog["label_3"]).fillna(0)
mix.plot.bar(figsize=(8, 4), title="Built pipe length per type"); plt.ylabel("m"); plt.tight_layout()
plt.savefig(OUT / "pipe_mix_vs_step1.png", dpi=150); plt.show()
mix

## 9. Save results – incl. the network losses for Step 4
`network_losses.csv` is the **link to oemof** (Step 4): total network heat loss in kW at the design hour and per year.
Because losses are modelled flow-independent (`l_factor = 0`), the hourly loss equals this kW value in every hour.

In [ ]:
built.to_csv(OUT / "pipes_results.csv", index=False)
producer_table.to_csv(OUT / "producers_results.csv")
comparison.to_csv(OUT / "comparison_with_step1.csv")
summary.to_csv(OUT / "summary.csv", header=["Step 2"])

network_losses = pd.DataFrame({
    "total_loss_kW": [built["losses"].sum()],
    "annual_loss_MWh": [built["losses"].sum() * 8760 / 1000],
    "built_length_m": [built["length"].sum()],
})
network_losses.to_csv(OUT / "network_losses.csv", index=False)
print(network_losses)
print("Saved to", OUT)

## 10. Questions to answer
✏️ Answer with numbers from sections 7–8.

1. **How did increasing the number of consumers and producers affect the network topology and pipe selection?** *(plots + pipe mix)*
2. **Did the optimizer choose different pipe types compared to Step 1? Why?** *(more types → closer match between required flow and `cap_max`; aggregated trunk flows)*
3. **Are some producers associated with substantially lower supplied capacity than others?** *(section 7.2 – explain the mechanism: cost, location, binding pipes 7.3)*
4. **How are total investment cost, operating cost and losses compared with Step 1?** *(section 8 – absolute and per kW of demand)*
5. Computational complexity: how did the problem size and solve time change?